# COMP 440 HW2: Whose Preferences Count?

This is the Colab notebook for HW2. Claude Code loads it into Colab from `colab/hw2.ipynb` in
your repository, so everyone starts from the same notebook. Changes made here are not saved back
to your repository, and that's fine: if Colab disconnects, Claude loads the notebook again. You
won't type much here. Claude
Code, on your laptop, adds, changes and runs the cells for you through the Colab MCP server, and
I've put a short note above each cell so you can follow what it does. The notebook needs a T4 GPU.

## Setup

Run these three cells in order whenever you open the notebook or reconnect. A reconnect starts
over on a new machine, so anything loaded before is gone.

### Check the GPU

This cell checks that the notebook has a GPU and prints which one, with the versions of PyTorch
and transformers. If it says there is no GPU, choose Runtime > Change runtime type > T4 GPU and
run it again.

In [ ]:
import torch
import transformers

print(f"torch {torch.__version__}, transformers {transformers.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook has no GPU. In Colab's menu, choose Runtime > Change runtime type > "
        "T4 GPU, then run this cell again.")
gpu = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu}, {memory_gb:.1f} GB of memory")
if "T4" not in gpu:
    print("This GPU is not a T4. The notebook should still work, but times and memory use "
          "will differ from the instructions.")

### Get the test items

This cell downloads the three shared test sets, 240 items in all, from the class repository into
`/content/hw2/items/`. It skips any file that is already there.

In [ ]:
import json
import os
import urllib.request

ITEMS_URL = "https://raw.githubusercontent.com/shilad/comp440-hw2/main/evals/shared"
WORK_DIR = "/content/hw2"
ITEMS_DIR = f"{WORK_DIR}/items"
FAMILIES = ["facts", "user_says_something_wrong", "emotional_social"]

os.makedirs(ITEMS_DIR, exist_ok=True)
items = {}
for family in FAMILIES:
    path = f"{ITEMS_DIR}/{family}.jsonl"
    if os.path.exists(path):
        print(f"{family}.jsonl is already in {ITEMS_DIR}, so it was not downloaded")
    else:
        url = f"{ITEMS_URL}/{family}.jsonl"
        try:
            urllib.request.urlretrieve(url, path + ".part")
        except OSError as e:
            raise RuntimeError(f"Could not download {url} ({e}). Run this cell again in a "
                               "minute. If it keeps failing, ask in #comp440-f26 on Slack.") from e
        os.replace(path + ".part", path)
    with open(path, encoding="utf-8") as f:
        items[family] = [json.loads(line) for line in f if line.strip()]

for family in FAMILIES:
    print(f"{family}: {len(items[family])} items")
print(f"total: {sum(len(v) for v in items.values())} items")

### Load the model

This cell downloads Qwen3-0.6B-Base from Hugging Face and puts it on the GPU, stored as 16-bit
numbers (float16). It takes about 20 seconds, and a warning that you have no Hugging Face token
is fine.

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

# transformers 4.56 renamed torch_dtype to dtype. Use the name this version knows.
major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
dtype_arg = "dtype" if (major, minor) >= (4, 56) else "torch_dtype"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **{dtype_arg: torch.float16}).to("cuda")
print(f"Loaded {MODEL_NAME}. GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

## Part 1

These cells go with Steps 2 and 5 of `instructions/part1.md`.

### Step 2: See what the model predicts next

`top_next_tokens` prints the five most likely next tokens after some text, with their
probabilities. To try another prompt, change the text in the last line and run the cell again.

In [ ]:
def top_next_tokens(text, k=5):
    """Print the k most likely next tokens after text, with their probabilities."""
    ids = tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        scores = model(**ids).logits[0, -1]
    probs = torch.softmax(scores.float(), dim=-1)
    top = torch.topk(probs, k)
    for p, i in zip(top.values, top.indices):
        print(f"{p.item():.3f}  {tok.decode(int(i))!r}")

top_next_tokens("Question: Which city is the capital of Canada?\nAnswer:")

### Step 5: Code for answering the test items

This cell only defines functions: `answer_items` gives every test item to a model and records
its answers, and `save_answers` zips them and downloads the zip. Part 2 uses both again for your
trained models.

In [ ]:
import time
import zipfile
from google.colab import files

PROMPT = "Question: {prompt}\nAnswer:"
MAX_NEW_TOKENS = {"facts": 120, "user_says_something_wrong": 120, "emotional_social": 200}
REPETITION_PENALTY = 1.15
BATCH_SIZE = 32

# A copy of the tokenizer for batches. It pads short prompts on the left, because the model
# writes on the right. It pads with a token the model never writes: padding with the
# end-of-text token would let the repetition penalty push the model away from stopping.
batch_tok = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
batch_tok.pad_token = "<|vision_pad|>"


def generate_batch(model, prompts, max_new_tokens):
    """Answer a list of prompts at once. Returns (raw text, number of new tokens) for each."""
    enc = batch_tok([PROMPT.format(prompt=p) for p in prompts], return_tensors="pt",
                    padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             repetition_penalty=REPETITION_PENALTY,
                             pad_token_id=batch_tok.pad_token_id,
                             eos_token_id=batch_tok.eos_token_id)
    results = []
    for row in out[:, enc["input_ids"].shape[1]:].tolist():
        # Keep the tokens before the end-of-text token, if the model wrote one.
        n = row.index(batch_tok.eos_token_id) if batch_tok.eos_token_id in row else len(row)
        results.append((batch_tok.decode(row[:n], skip_special_tokens=True), n))
    return results


def answer_items(model, model_id, items, batch_size=BATCH_SIZE):
    """Give every test item to the model. Returns one row per item, and notes on how."""
    rows, start, first_size = [], time.time(), batch_size
    for family, todo in items.items():
        max_new = MAX_NEW_TOKENS[family]
        done = 0
        while done < len(todo):
            batch = todo[done:done + batch_size]
            try:
                results = generate_batch(model, [it["prompt"] for it in batch], max_new)
            except torch.cuda.OutOfMemoryError:
                results = None
            if results is None:
                if batch_size == 1:
                    raise RuntimeError("Out of GPU memory even with one item at a time. Choose "
                                       "Runtime > Restart session, then run the cells again.")
                batch_size //= 2
                torch.cuda.empty_cache()
                print(f"Out of GPU memory. Trying again with batches of {batch_size}.")
                continue
            for it, (raw, n) in zip(batch, results):
                rows.append({
                    "id": it["id"],
                    "model": model_id,
                    # The base model often goes on to invent the next question. Cut it there.
                    "response": raw.split("\nQuestion:")[0].strip(),
                    "raw": raw,
                    "new_tokens": n,
                    "max_new_tokens": max_new,
                    "hit_limit": n == max_new,
                })
            done += len(batch)
            print(f"{family}: {done} of {len(todo)} done, {time.time() - start:.0f} s")

    seconds = time.time() - start
    hits = sum(r["hit_limit"] for r in rows)
    print(f"Finished {len(rows)} items in {seconds:.0f} s. {hits} ran to the length limit.")
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "a CPU"
    sizes = f"{first_size}" if batch_size == first_size else f"{first_size}, then {batch_size}"
    notes = (f"Made {time.strftime('%Y-%m-%d')} with the COMP 440 HW2 Colab notebook: "
             f"{len(rows)} answers in {seconds:.0f} s on {gpu}, weights in "
             f"{str(model.dtype).replace('torch.', '')}, torch {torch.__version__}, "
             f"transformers {transformers.__version__}. Greedy decoding (do_sample=False) with "
             f"repetition_penalty {REPETITION_PENALTY}, in batches of {sizes}, padded on the left. "
             "The response is the raw text cut where the model starts a new line with "
             "'Question:', with spaces trimmed from both ends.")
    return rows, notes


def save_answers(rows, notes, label, description, charter="none"):
    """Save the answers in the format of a run folder, zip them and download the zip."""
    model_id = rows[0]["model"]
    name = f"hw2-{model_id}-answers"
    folder = f"{WORK_DIR}/{name}"
    os.makedirs(f"{folder}/responses", exist_ok=True)
    with open(f"{folder}/responses/{model_id}.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(f"{folder}/models.json", "w", encoding="utf-8") as f:
        json.dump([{"id": model_id, "label": label, "description": description,
                    "charter": charter}], f, indent=2)
    with open(f"{folder}/generation-notes.txt", "w", encoding="utf-8") as f:
        f.write(notes + "\n")
    with open(f"{folder}/README.txt", "w", encoding="utf-8") as f:
        f.write(f"""Answers from the COMP 440 HW2 Colab notebook, model "{model_id}".

responses/{model_id}.jsonl   one line per test item: id, model, response, raw, new_tokens,
    max_new_tokens, hit_limit
models.json   what the model is, for init-run's --models-json
generation-notes.txt   how the answers were made, for init-run's --generation-notes

To make a run from this folder, run this from the repository's folder, with FOLDER the path
to this folder:

python3 evals/shared/grade_tools.py init-run --root evals/runs --name SLUG \\
    --title "..." --description "..." --items evals/shared \\
    --responses FOLDER/responses/{model_id}.jsonl --models-json FOLDER/models.json \\
    --generation-notes "$(cat FOLDER/generation-notes.txt)"
""")
    zip_path = f"{WORK_DIR}/{name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for part in [f"responses/{model_id}.jsonl", "models.json", "generation-notes.txt",
                     "README.txt"]:
            z.write(f"{folder}/{part}", f"{name}/{part}")
    print(f"Saved {len(rows)} answers in {zip_path}. Downloading it now.")
    files.download(zip_path)

### Step 5: Answer every test item with the base model

This gives all 240 items to the base model in batches and prints its progress. It should take
about a minute on a T4.

In [ ]:
base_rows, base_notes = answer_items(model, "base", items)
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
print("The first answer:")
print(json.dumps(base_rows[0], ensure_ascii=False, indent=1))

### Step 5: Save and download the answers

This saves the answers as `responses/base.jsonl`, zips them into `hw2-base-answers.zip` and
downloads the zip to your laptop. The first time, Chrome asks near the address bar whether Colab
may download files; allow it.

In [ ]:
save_answers(base_rows, base_notes, label="Base",
             description="Qwen3-0.6B-Base, the pretrained model with no further training.")

### Free the GPU memory

This takes the model off the GPU, so the memory is free for training in Part 2. To use the model
again, run the Load the model cell again.

In [ ]:
import gc

model = None
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.2f} GB")

## Part 2

The cells for training your two models will go here.